# Figure 1 — SHERLOCK on genome-scale Perturb-seq (Replogle)

Every code cell below is copied verbatim from an existing analysis notebook; nothing
here is new analysis. Each source section starts with a small `[reproducibility glue]`
cell that changes into the source notebook's folder (so its relative data and results
paths still work) and resets the sherlock config. Cells that are not labelled with a
panel are the prerequisites that panel needed in the source notebook (data loading,
model loading, helper definitions). Panels are produced separately rather than laid out
as in the manuscript. Anything that could not be found in the current notebooks is
marked **FLAG**.

In [ ]:
import os
REPRO_DIR = os.path.abspath('')

> **FLAG — needs further investigation.** **Fig. 1a–c** are schematics (overview, plate diagram, functionalities); there's no code for them.

> **FLAG — needs further investigation.** **Fig. 1d (left, middle)**: UMAP of the original gene-expression (PCA) space coloured by pathway and by Leiden cluster. No cell in the current notebooks computes a PCA/Leiden UMAP of Replogle.

## Source: `notebooks/benchmarking.ipynb`

Cells copied verbatim (source cell indices): 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17. Cell 9 loads each saved run from `results/benchmarking/models/` and only trains runs whose checkpoint is missing.

In [ ]:
# [reproducibility glue] run from the source notebook's directory so its relative paths resolve
os.chdir(os.path.join(REPRO_DIR, '../notebooks/.'))
if 'slk' in globals():
    slk.configs.reset_config()

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import torch
from matplotlib.colors import to_hex
from scipy.optimize import linear_sum_assignment

sys.path.append('../..')
import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib as mpl

# ── Journal figure style ────────────────────────────────────────────────
RC = {
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':             8,
    'axes.labelsize':        8,
    'axes.titlesize':        9,
    'xtick.labelsize':       7,
    'ytick.labelsize':       7,
    'legend.fontsize':       6,
    'legend.title_fontsize': 7,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.linewidth':        0.75,
    'xtick.major.width':     0.75,
    'ytick.major.width':     0.75,
    'xtick.major.size':      3,
    'ytick.major.size':      3,
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
    'svg.fonttype': 'none',
}
mpl.rcParams.update(RC)

In [ ]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_RUNS      = 5
NUM_EPOCHS  = 200
VALIDATE_EVERY = 50
BATCH_SIZE  = 4096
LATENT_DIM  = 16
PATIENCE    = 20
RESULTS_DIR = "results/benchmarking"
N_CLUSTERS  = 8

# Metrics tracked for scoring and box plots
METRIC_COLS = [
    'cf_ate_pearson_r',
    'cluster_f1',
    'cluster_ari',
    'cluster_ami',
    'cluster_silhouette',
    'cluster_linear_probe',
    'cluster_knn_purity',
]

# (model_key, display_name, method-specific kwargs)
# sherlock (vae) kwargs mirror the tuned hyperparameters in "demo copy.ipynb"
# (cov_lambda / H_lambda / gate_row_repulsion_lambda disabled, ce_lambda raised to 20)
METHODS = [
    ("vae",           "sherlock",      {
        "l0_lambda": 1e-1,
        "ce_lambda": 20.,
        "n_epochs_l0_warmup": 300,
    },  500),
    ("svae",          "sVAE",          {"sparse_mask_penalty": 10.0},  NUM_EPOCHS),
    ("cvae",          "cVAE",          {},  NUM_EPOCHS),
    ("contrastivevi", "contrastiveVI", {},  NUM_EPOCHS),
    ("scgen",         "scgen",         {},  NUM_EPOCHS),
]

####TODO CHECK THIS#####
METHODS = [
    ("vae",           "sherlock",      {
        "l0_lambda": 1e-1,
        "ce_lambda": 20.,
        "n_epochs_l0_warmup": 300,
    }, 500),
    ("svae",          "sVAE",          {
        "sparse_mask_penalty": 4.366473592979633,
        "beta": 0.614252574393396,
        "n_hidden": 251,
        "n_layers": 3,
        "dropout_rate": 0.28184968246925673,
        "lr": 0.00331348361550895,
    }, NUM_EPOCHS),
    ("cvae",          "cVAE",          {
        "beta": 3.2226582291306802,
        "n_hidden": 67,
        "n_layers": 1,
        "dropout_rate": 0.29147206450401036,
        "lr": 0.006205861500406846,
    }, NUM_EPOCHS),
    ("contrastivevi", "contrastiveVI", {
        "wasserstein_penalty": 4.399492045282824,
        "n_hidden": 94,
        "n_layers": 2,
        "dropout_rate": 0.2902495045654139,
        "lr": 0.0005218479707033748,
    }, NUM_EPOCHS),
    ("scgen",         "scgen",         {}, NUM_EPOCHS),
]

os.makedirs(f"{RESULTS_DIR}/models",   exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures",  exist_ok=True)

print(f"Device: {DEVICE}  Results: {RESULTS_DIR}")

## Data loading and ground-truth ATE

In [ ]:
data = slk.datasets.replogle()

slk.pp.treat_effect(
    data,
    label_col='pert',
    control_label='NTC',
    method='perturbseq',
    inplace=True,
    key='treat_effect',
)
treat_effect_adata = data.uns['treat_effect']
print(f"Cells: {data.n_obs}  Genes: {data.n_vars}  Perturbations: {data.obs['pert'].nunique()}")

## Pathway annotations for clustering evaluation

In [ ]:
pathways = data.uns['pathways']

all_genes_filtered = [
    gene
    for genes in pathways.values()
    for gene in genes
    if gene != 'LSM5'
]

gene_to_pathway = {
    gene: pathway
    for pathway, genes in pathways.items()
    for gene in genes
    if gene != 'LSM5'
}
pathway_df_indexed = pd.DataFrame.from_dict(
    gene_to_pathway, orient='index', columns=['pathway']
)
pathway_df_indexed.index.name = 'gene'

print(f"Pathway genes: {len(all_genes_filtered)}  Pathways: {len(pathways)}")

## Benchmark loop

Each method is trained `N_RUNS` times independently. The best-scoring run
per method is saved to `results/models/`. Metrics are computed after training
using the proper counterfactual ATE protocol; the clustering cut is chosen
automatically via the elbow method.

In [ ]:
records         = []
best_run_out    = {}   # method → out dict of best single run
best_run_model  = {}   # method → model of best single run
best_run_score  = {}   # method → avg metric score of best single run
best_run_path   = {}   # method → run_x.pth of best single run

for model_key, display_name, extra_kwargs, EPOCHS, in METHODS:
    print(f"\n{'='*60}\n{display_name}  ({N_RUNS} runs)\n{'='*60}")
    best_run_score[display_name] = -np.inf

    # Each method gets its own folder; individual runs are checkpointed there
    # so a re-executed notebook reloads finished runs instead of retraining.
    model_dir = f"{RESULTS_DIR}/models/{display_name}"
    os.makedirs(model_dir, exist_ok=True)

    for run in range(N_RUNS):
        print(f"\n  -- {display_name} run {run + 1}/{N_RUNS} --")

        run_path = f"{model_dir}/run_{run}.pth"

        if os.path.exists(run_path):
            print(f"    Found checkpoint → {run_path}, loading instead of retraining")
            model = slk.tl.load_results(run_path).to(DEVICE)
            out = {"model": model, "param_store": None}
        else:
            out = slk.tl.run(
                data,
                model=model_key,
                batch_size=BATCH_SIZE,
                num_epochs=EPOCHS,
                device=DEVICE,
                latent_dim=LATENT_DIM,
                patience=PATIENCE,
                validate_every=VALIDATE_EVERY,
                seed=run,
                **extra_kwargs,
            )
            model = out['model']
            slk.tl.save_results(out, run_path)
            print(f"    Saved → {run_path}")

        metrics = slk.tl.evaluate_model(
            model,
            data,
            treat_effect_adata,
            pathway_df_indexed,
            all_genes_filtered,
            n_clusters=N_CLUSTERS,
            device=DEVICE,
            use_rho=False,
        )

        # Track best run for this method
        run_avg = float(metrics['cf_ate_pearson_r'])
        if run_avg > best_run_score[display_name]:
            best_run_score[display_name] = run_avg
            best_run_out[display_name]   = out
            best_run_model[display_name] = model
            best_run_path[display_name]  = run_path

        metrics['method'] = display_name
        metrics['run']    = run
        records.append(metrics)

        for k, v in metrics.items():
            if k not in ('method', 'run'):
                val_str = f"{v:.4f}" if np.isfinite(float(v)) else "NaN"
                print(f"    {k:30s} = {val_str}")

    # Save best run model as an exact copy of its own run_x.pth. Copying (vs.
    # re-saving from memory) guarantees best.pth carries the best run's
    # state_dict AND its Pyro param store (sigma_fac, …). The store is not in
    # state_dict, and the *global* Pyro store reflects whichever run ran last,
    # not necessarily the best — so we never rely on it.
    import shutil
    save_path = f"{model_dir}/best.pth"
    shutil.copyfile(best_run_path[display_name], save_path)
    print(f"\n  Saved best {display_name} model → {save_path} "
          f"(copied from {os.path.basename(best_run_path[display_name])})")

results_df = pd.DataFrame(records)
print("\nDone.")

## Results summary

In [ ]:
summary = (
    results_df
    .groupby('method')[METRIC_COLS]
    .agg(['mean', 'std'])
    .round(4)
)
# Preserve METHODS order
method_order = [m[1] for m in METHODS]
summary = summary.reindex([m for m in method_order if m in summary.index])
print(summary.to_string())

## Box plots

### Fig. 1g — Replogle benchmarking (ATE, F1, ARI, AMI, silhouette)
*(source cell 13)*

In [ ]:
# Toggle significance brackets (SHERLOCK vs each baseline, per metric column).
# Welch's t-test on the per-run values, Benjamini-Hochberg FDR across the whole
# family, drawn ggsignif-style as ns / * / ** / *** (same as norman_figure Panel A).
SIG = True

CLR_SHERLOCK    = '#2166ac'
CLR_GEARS       = '#d73027'

METHOD_PALETTE = {
    'SHERLOCK':      CLR_SHERLOCK,
    'cVAE':          '#33a02c',
    'sVAE':          '#ff7f00',
    'contrastiveVI': '#6a3d9a',
    'GEARS':         CLR_GEARS,
}

label_map = {
    'cf_ate_pearson_r':   'ATE',
    'cluster_f1':         'F1',
    'cluster_ari':        'ARI',
    'cluster_ami':        'AMI',
    'cluster_silhouette': 'Silhouette',
    # 'cluster_linear_probe': 'Linear\nProbe',
    # 'cluster_knn_purity': 'kNN\nPurity',
}

METRIC_COLS_PLOT = [
    m for m in METRIC_COLS
    if m not in ['cluster_linear_probe', 'cluster_knn_purity']
]

# Display names matched to METHOD_PALETTE keys (sherlock -> SHERLOCK)
method_display = {
    'sherlock':      'SHERLOCK',
}
method_order = [method_display.get(m[1], m[1]) for m in METHODS]

df_melted = results_df.melt(
    id_vars='method', value_vars=METRIC_COLS_PLOT, var_name='metric', value_name='value'
)
df_melted['method'] = df_melted['method'].map(lambda m: method_display.get(m, m))
df_melted['metric_label'] = pd.Categorical(
    df_melted['metric'].map(label_map),
    categories=[label_map[m] for m in METRIC_COLS_PLOT],
    ordered=True,
)
df_melted['method'] = pd.Categorical(
    df_melted['method'], categories=method_order, ordered=True
)
df_melted['value'] = df_melted['value'].clip(lower=0)

# scgen has no ATE result to show
df_melted = df_melted[~((df_melted['method'] == 'scgen') & (df_melted['metric'] == 'cf_ate_pearson_r'))]

# Fall back to gray for any method not in METHOD_PALETTE (scgen is colored red)
palette = {m: METHOD_PALETTE.get(m, '#999999') for m in method_order}
palette['scgen'] = CLR_GEARS

# Taller when SIG is on so the stacked brackets have headroom above the boxes.
fig, ax = plt.subplots(figsize=(6.5*0.75, (2.7 if SIG else 2)*0.75))

# NOTE: stripplot's dodge always spans a fixed width of 0.8 (it has no `width`
# kwarg), so the boxplot must also dodge over width=0.8 for box/point centers
# to line up -- otherwise outer hue levels (e.g. scgen) drift off to the side.
# `gap` shrinks the box's visual thickness without moving its dodge position.
sns.boxplot(
    data=df_melted,
    x='metric_label',
    y='value',
    hue='method',
    hue_order=method_order,
    palette=palette,
    width=0.8,
    gap=0.25,
    linewidth=0.6,
    showfliers=False,
    boxprops=dict(alpha=0.55),
    ax=ax,
)

sns.stripplot(
    data=df_melted,
    x='metric_label',
    y='value',
    hue='method',
    hue_order=method_order,
    palette=palette,
    dodge=True,
    jitter=0.1,
    size=3,
    linewidth=0.3,
    edgecolor='white',
    legend=False,
    ax=ax,
)

ax.set_xlabel('')
ax.set_ylabel('Score')
ax.legend(title='Method', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_ylim(0, 1.0)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')

# Separator between ATE and clustering metrics
ax.axvline(0.5, color='lightgray', linewidth=1.0, linestyle=':')

# ── Significance: SHERLOCK vs every other method, per metric column ────────
# Brackets are drawn above the y=1.0 axis (clip_on=False) so the score axis
# stays a clean 0-1; savefig uses bbox_inches='tight' so they aren't clipped.
if SIG:
    from scipy import stats as _stats
    _sig_metrics = METRIC_COLS_PLOT                        # raw metric names, in column order
    _sig_others  = [m for m in method_order if m != 'SHERLOCK']

    def _mvals(_mk, _meth):
        return df_melted[(df_melted['metric'] == _mk) &
                         (df_melted['method'] == _meth)]['value'].dropna().to_numpy()

    _comps = []                                            # [col_i, other_method, p]
    for _i, _mk in enumerate(_sig_metrics):
        _s = _mvals(_mk, 'SHERLOCK')
        for _o in _sig_others:
            _ov = _mvals(_mk, _o)
            if len(_s) < 2 or len(_ov) < 2:
                continue
            _comps.append([_i, _o, _stats.ttest_ind(_s, _ov, equal_var=False).pvalue])

    # Benjamini-Hochberg across the whole family
    _pv = np.array([c[2] for c in _comps])
    _ord = np.argsort(_pv); _mtot = len(_pv); _adj = np.empty(_mtot); _prev = 1.0
    for _k in range(_mtot - 1, -1, -1):
        _ix = _ord[_k]; _prev = min(_prev, _pv[_ix] * _mtot / (_k + 1)); _adj[_ix] = min(_prev, 1.0)
    for _c, _pa in zip(_comps, _adj):
        _c.append(_pa)

    def _stars(_p):
        return '***' if _p < 1e-3 else '**' if _p < 1e-2 else '*' if _p < 0.05 else 'ns'

    # dodge geometry (boxplot width=0.8, one box per method)
    _n_hue, _box_w = len(method_order), 0.8
    def _xpos(_i, _meth):
        _j = method_order.index(_meth)
        return _i - _box_w / 2 + _box_w / (2 * _n_hue) + _j * (_box_w / _n_hue)

    _y0   = 1.03                                           # first bracket just above the y=1.0 axis
    _step = 0.075                                          # gap between stacked brackets
    _tick = 0.012                                          # bracket end-tick height
    _lbl_h = 0.028                                         # ink height reserved for a star label
    # Asterisk glyphs sit high in the text box; 'ns' letters sit on the baseline.
    # These per-label nudges land the ink at the same height above every bracket
    # line so all labels read as equal-height regardless of glyph.
    _lbl_dy = {'ns': 0.006}
    _ytop = _y0
    for _i in range(len(_sig_metrics)):
        _col = sorted([c for c in _comps if c[0] == _i],
                      key=lambda c: method_order.index(c[1]))   # nearest method lowest
        for _lvl, (_, _o, _p, _pa) in enumerate(_col):
            _x1, _x2 = _xpos(_i, 'SHERLOCK'), _xpos(_i, _o)
            _y = _y0 + _lvl * _step
            ax.plot([_x1, _x1, _x2, _x2], [_y, _y + _tick, _y + _tick, _y],
                    lw=0.5, c='0.35', clip_on=False)
            _lbl = _stars(_pa)
            ax.text((_x1 + _x2) / 2, _y + _tick + _lbl_dy.get(_lbl, -0.014), _lbl,
                    ha='center', va='bottom', fontsize=5, color='0.2', clip_on=False)
            _ytop = max(_ytop, _y + _tick + _lbl_h)

    # Title above the tallest bracket (axis is 0-1, so a data value == axes frac).
    ax.set_title('Replogle Benchmarking Results', y=_ytop + 0.04)
else:
    ax.set_title('Replogle Benchmarking Results', pad=6)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/figures/benchmark_boxplot.svg', bbox_inches='tight')
plt.savefig(f'{RESULTS_DIR}/figures/benchmark_boxplot.pdf', bbox_inches='tight')
plt.savefig(f'{RESULTS_DIR}/figures/benchmark_boxplot.png', bbox_inches='tight')
plt.show()

### Fig. 1d (right) — SHERLOCK latent space UMAP coloured by pathway (this cell makes one UMAP per method; the SHERLOCK one is the manuscript panel)
*(source cell 17)*

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC   = slk.configs.get_config('ntc_label')

for method_name, model in best_run_model.items():
    print(f"\n{'='*60}\nGenerating figure for best {method_name} model\n{'='*60}")

    # Re-eval the best model for this method
    slk.tl.eval(model, data, obsm_key='z', uns_key='results', device=DEVICE)

    # UMAP on non-NTC cells
    sub = data[data.obs[p_key] != NTC].copy()
    sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
    sc.tl.umap(sub)

    # Cluster z_corr on pathway genes
    z_corr  = np.asarray(data.uns['results']['z_corr'])
    z_perts = np.asarray(data.uns['results']['z_perts'])

    pert_to_idx = {name: i for i, name in enumerate(z_perts)}
    valid_genes = [g for g in all_genes_filtered if g in pert_to_idx]
    gene_ids    = [pert_to_idx[g] for g in valid_genes]

    cov_subset = z_corr[np.ix_(gene_ids, gene_ids)]
    cov_subset = np.clip(cov_subset, -1.0, 1.0)
    cov_subset = 0.5 * (cov_subset + cov_subset.T)
    np.fill_diagonal(cov_subset, 1.0)

    groups = slk.tl.cluster_rho(data, t=N_CLUSTERS, rho_corr=cov_subset, criterion='maxclust', show=False)

    # Hungarian alignment of cluster → pathway labels
    clusters_df = pd.DataFrame({'gene': valid_genes, 'cluster': groups})
    pw_df = pathway_df_indexed.loc[pathway_df_indexed.index.isin(valid_genes)].reset_index()
    if 'gene' not in pw_df.columns:
        pw_df = pw_df.rename(columns={pathway_df_indexed.index.name or 'index': 'gene'})

    df_align  = pd.merge(clusters_df, pw_df, on='gene', how='inner')
    y_true    = df_align['pathway'].astype(str).values
    y_pred    = df_align['cluster'].astype(str).values
    true_lbl  = np.unique(y_true)
    pred_lbl  = np.unique(y_pred)

    C_mat = (
        pd.crosstab(df_align['pathway'].astype(str), df_align['cluster'].astype(str))
        .reindex(index=true_lbl, columns=pred_lbl, fill_value=0)
    )
    row_ind, col_ind = linear_sum_assignment(C_mat.values.max() - C_mat.values)
    mapping = {pred_lbl[j]: true_lbl[i] for i, j in zip(row_ind, col_ind)}
    df_align['mapped_pathway'] = [mapping.get(str(cl)) for cl in y_pred]

    # Subset z_sub to clustered genes and attach pathway labels
    clustered_genes = df_align['gene'].unique()
    z_sub = sub[sub.obs[p_key].isin(clustered_genes)].copy()

    gene_to_true   = dict(zip(df_align['gene'], df_align['pathway']))
    gene_to_mapped = dict(zip(df_align['gene'], df_align['mapped_pathway']))
    z_sub.obs['true_pathway']   = z_sub.obs[p_key].map(gene_to_true)
    z_sub.obs['mapped_pathway'] = z_sub.obs[p_key].map(gene_to_mapped)
    z_sub = z_sub[z_sub.obs['mapped_pathway'].notna()].copy()

    # Shared categorical palette across both columns
    all_cats  = pd.Index(np.unique(
        list(z_sub.obs['true_pathway'].dropna().unique()) +
        list(z_sub.obs['mapped_pathway'].dropna().unique())
    ).astype(str))
    cat_dtype = pd.api.types.CategoricalDtype(categories=all_cats, ordered=False)
    z_sub.obs['true_pathway']   = z_sub.obs['true_pathway'].astype('string').astype(cat_dtype)
    z_sub.obs['mapped_pathway'] = z_sub.obs['mapped_pathway'].astype('string').astype(cat_dtype)

    cmap_col       = plt.cm.get_cmap('tab20', len(all_cats))
    shared_palette = {cat: to_hex(cmap_col(i)) for i, cat in enumerate(all_cats)}
    z_sub.uns['true_pathway_colors']   = [shared_palette[c] for c in z_sub.obs['true_pathway'].cat.categories]
    z_sub.uns['mapped_pathway_colors'] = [shared_palette[c] for c in z_sub.obs['mapped_pathway'].cat.categories]

    # Plot and save
    fig = sc.pl.umap(
        z_sub,
        color=['true_pathway', 'mapped_pathway'],
        wspace=0.4,
        show=False,
        return_fig=True,
    )
    fig.suptitle(f'Best {method_name} model', y=1.02, fontsize=12)
    save_path = f'{RESULTS_DIR}/figures/{method_name}_best_umap.svg'
    fig.savefig(save_path, bbox_inches='tight')
    print(f"  Saved → {save_path}")
    plt.show()

> **FLAG — needs further investigation.** **Fig. 1e**: the manuscript's correlation matrix has *Reference* and *Inferred* pathway annotation bars. No current cell draws those bars. The closest source, the plain `slk.pl.plot_corr` clustermap of the learned perturbation correlation, is included below.

## Source: `notebooks/demo copy.ipynb (commit bd8f071, before it was rewritten as demo.ipynb)`

Cells copied verbatim (source cell indices): 0, 1, 2, 3, 4, 5, 10, 11, 15, 18, 24, 25, 26, 29, 30, 31, 32, 33, 39, 40. Training/saving cells (6–9), the very large `plot_ev_sig` heatmap (17), and three cells that were run out of order (36–38) are not copied. Cell 40 needs `gseapy` and internet access (Enrichr).

In [ ]:
# [reproducibility glue] run from the source notebook's directory so its relative paths resolve
os.chdir(os.path.join(REPRO_DIR, '../notebooks/.'))
if 'slk' in globals():
    slk.configs.reset_config()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

import seaborn as sns
import matplotlib.pyplot as plt

import torch

#Make sure to path to wherever sherlock is located
sys.path.append('../..')
import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
slk.configs.show()
slk.configs.set_config('ntc_label', 'non-targeting')
print('\n')
slk.configs.show()
slk.configs.reset_config()
slk.configs.get_config('ntc_label')

In [ ]:
data = slk.datasets.replogle()

'''Alternatively for your own data:
data = sc.read_h5ad(PATH_TO_YOUR_DATA)
slk_prepare_data(data, pert_key=PERT_KEY, ntc_label=NTC_LABEL, treatment_key=TREATMENT_KEY|None, inplace=True)
'''

In [ ]:
perturbations = set(np.unique(data.obs[slk.configs.get_config('pert_key')].values))
print("Perturbations:", len(perturbations))

pathways = []
for pathway in data.uns['pathways'].values():
    pathways.append(set(pathway))
sum([len(pathways[i]) for i in range(len(pathways))])
print("Pathways:", len(pathways))

overlap = any(len(set(pathways[i]) & set(pathways[j])) > 0 for i in range(len(pathways)) for j in range(i + 1, len(pathways)))
print("Overlap exists between pathways:" if overlap else "No overlap between pathways.")

In [ ]:
slk.pp.treat_effect(
    data,
    label_col = 'pert',
    control_label = 'NTC',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)

In [ ]:
model = slk.tl.load_results('./results/benchmarking/models/sherlock/best.pth')

In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results")

In [ ]:
slk.tl.ev_sig(data)

### Fig. 1e (partial) — learned perturbation correlation matrix, without annotation bars
*(source cell 18)*

In [ ]:
slk.pl.plot_corr(data)

In [ ]:
p_key = slk.configs.get_config('pert_key')

dataset = slk.tl.PerturbMatchingDataset(data)
n_perts    = len(dataset.perturbation_dict)
latent_dim = data.obsm['z'].shape[1]

z_bar = np.zeros((n_perts, latent_dim))
for pert, pidx in dataset.perturbation_dict.items():
    mask = data.obs[p_key] == pert
    if mask.any():
        z_bar[pidx] = data.obsm['z'][mask].mean(0)

C = np.corrcoef(z_bar)

In [ ]:
C = data.uns['results']['z_corr']

#Specific subsetting for replogle. Otherwise just use C directly
all_genes = [gene for genes in data.uns['pathways'].values() for gene in genes]
dataset = slk.tl.PerturbMatchingDataset(data)
gene_ids = [dataset.perturbation_dict[g] for g in all_genes]
cov_subset = C[np.ix_(gene_ids, gene_ids)].clip(-1, 1)

In [ ]:
groups = slk.tl.cluster_rho(data, t=1, criterion='distance', method='ward', rho_corr=cov_subset, show=True)

In [ ]:
from collections import Counter

# Flatten all genes from all pathways
#all_genes = [gene for genes in data.uns['pathways'].values() for gene in genes]

# Count how many times each gene appears
gene_counts = Counter(all_genes)

# Find genes that appear in more than one pathway
shared_genes = [gene for gene, count in gene_counts.items() if count > 1]

if shared_genes:
    print(f"⚠️ {len(shared_genes)} genes are shared across pathways:")
    print(shared_genes[:20])  # show first 20 for inspection
else:
    print("✅ Each gene appears in exactly one pathway.")


#pert df
pathway_genes = np.concatenate([i for i in list(data.uns['pathways'].values())])
pathway_genes = pathway_genes[pathway_genes != 'LSM5']
print(len(pathway_genes))
print(len(all_genes))
perts = pd.DataFrame(groups, index=all_genes, columns=['cluster'])
perts = perts.loc[pathway_genes]

#pathway df
gene_to_pathway = {
    gene: pathway
    for pathway, genes in data.uns['pathways'].items()
    for gene in genes if gene != 'LSM5'
}
pathway_df = pd.DataFrame.from_dict(gene_to_pathway, orient="index", columns=["cluster"])


perts = perts.drop(shared_genes, errors="ignore")
pathway_df = pathway_df.drop(shared_genes, errors="ignore")

print("After removal:")
print("perts size     :", perts.shape)
print("pathway_df size:", pathway_df.shape)

In [ ]:
pathway_df['gene'] = pathway_df.index.copy()
pathway_df.columns = ['pathway', 'gene']
pathway_df

In [ ]:
# Align on the same gene index
perts_aligned = perts.loc[pathway_df.index]

perts_aligned['gene'] = perts_aligned.index.copy()
perts_aligned

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score, adjusted_mutual_info_score,
    homogeneity_completeness_v_measure, silhouette_score,
    precision_score, recall_score, f1_score,
)
from sklearn.metrics.cluster import contingency_matrix
from scipy.optimize import linear_sum_assignment
from scipy.stats import hypergeom

try:
    from statsmodels.stats.multitest import multipletests
    _HAS_SM = True
except Exception:
    _HAS_SM = False


def align_and_summarize(
    clusters_df: pd.DataFrame,
    pathways_df: pd.DataFrame,
    gene_col: str = "gene",
    cluster_col: str = "cluster",
    pathway_col: str = "pathway",
    min_pathway_size: int = 1,
    gene_embeddings: pd.DataFrame | None = None,  # index=gene, cols=latent dims
):
    # 1) Merge on gene
    df = pd.merge(
        clusters_df[[gene_col, cluster_col]],
        pathways_df[[gene_col, pathway_col]],
        on=gene_col,
        how="inner",
        validate="one_to_one"
    ).dropna(subset=[cluster_col, pathway_col]).copy()

    # Optional: filter tiny pathways to stabilize Hungarian/enrichment
    if min_pathway_size > 1:
        keep_path = df[pathway_col].value_counts()
        keep_path = keep_path[keep_path >= min_pathway_size].index
        df = df[df[pathway_col].isin(keep_path)].copy()

    # Cast to string for safe indexing
    y_true = df[pathway_col].astype(str).values
    y_pred = df[cluster_col].astype(str).values
    true_labels = np.unique(y_true)
    pred_labels = np.unique(y_pred)

    # 2) Contingency (rows=pathways, cols=clusters)
    C = pd.crosstab(df[pathway_col].astype(str), df[cluster_col].astype(str))
    contingency_df = C.reindex(index=true_labels, columns=pred_labels, fill_value=0)

    # 3) Hungarian alignment to maximize correct assignments (sum of diagonal)
    M = contingency_df.values
    row_ind, col_ind = linear_sum_assignment(M.max() - M)  # maximize
    mapping = {pred_labels[j]: true_labels[i] for i, j in zip(row_ind, col_ind)}

    # Assign any unmapped clusters to their majority pathway (fallback)
    for cl in pred_labels:
        if cl not in mapping:
            cl_mask = df[cluster_col].astype(str) == cl
            if cl_mask.any():
                mapping[cl] = df.loc[cl_mask, pathway_col].mode().iloc[0]

    # 4) Remap clusters → pathways and compute metrics
    mapped = np.array([mapping.get(cl, None) for cl in y_pred], dtype=object)
    is_match = mapped == y_true
    accuracy = float(is_match.mean())

    # Global F1 / precision / recall (macro-averaged over pathways)
    precision = precision_score(y_true, mapped, average="macro", zero_division=0)
    recall    = recall_score(y_true, mapped, average="macro", zero_division=0)
    f1        = f1_score(y_true, mapped, average="macro", zero_division=0)

    # Label-based external clustering metrics
    ari = adjusted_rand_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y_true, y_pred)

    # Silhouette score (requires embeddings)
    sil = None
    if gene_embeddings is not None:
        genes_in_df = df[gene_col].values
        emb = gene_embeddings.loc[genes_in_df].values
        sil = silhouette_score(emb, mapped, metric='euclidean')

    # 5) Per-cluster purity & recall
    purity_rows = []
    for cl, g in df.groupby(cluster_col):
        majority = g[pathway_col].mode().iat[0]
        purity = (g[pathway_col] == majority).mean()
        pathway_total = (df[pathway_col] == majority).sum()
        recall_cl = (g[pathway_col] == majority).sum() / pathway_total
        purity_rows.append({
            "cluster": str(cl),
            "size": len(g),
            "mapped_pathway": mapping.get(str(cl), None),
            "purity": purity,
            "recall": recall_cl,
        })
    purity_df = pd.DataFrame(purity_rows).sort_values("cluster").set_index("cluster")

    # 6) Normalized contingency for plotting
    contingency = contingency_df.copy()
    contingency_row_norm = contingency.div(contingency.sum(axis=1).replace(0, np.nan), axis=0)
    contingency_col_norm = contingency.div(contingency.sum(axis=0).replace(0, np.nan), axis=1)

    # 7) Pathway↔cluster enrichment (hypergeometric with FDR)
    N = len(df)
    path_sizes = contingency.sum(axis=1)
    cl_sizes = contingency.sum(axis=0)
    records = []
    for p in contingency.index:
        K = int(path_sizes.loc[p])
        for c in contingency.columns:
            n = int(cl_sizes.loc[c])
            k = int(contingency.loc[p, c])
            pval = hypergeom.sf(k - 1, N, K, n) if k > 0 else 1.0
            records.append({"pathway": p, "cluster": c, "k": k, "K": K, "n": n, "N": N, "pval": float(pval)})

    enrich_df = pd.DataFrame(records)
    if _HAS_SM and len(enrich_df) > 0:
        enrich_df["qval"] = multipletests(enrich_df["pval"].values, method="fdr_bh")[1]
    else:
        m = max(len(enrich_df), 1)
        enrich_df["qval"] = np.minimum(enrich_df["pval"] * m, 1.0)

    with np.errstate(divide="ignore"):
        enrich_df["neglog10q"] = -np.log10(np.clip(enrich_df["qval"].values, 1e-300, 1.0))

    # 8) Attach mapped labels into the merged table
    df["mapped_pathway"] = mapped
    df["is_match"] = is_match

    return {
        "merged": df[[gene_col, cluster_col, pathway_col, "mapped_pathway", "is_match"]].rename(
            columns={gene_col: "gene", cluster_col: "cluster", pathway_col: "pathway"}
        ),
        "mapping": mapping,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "ari": ari,
        "ami": ami,
        "homogeneity": homogeneity,
        "completeness": completeness,
        "v_measure": v_measure,
        "silhouette": sil,
        "purity_df": purity_df,
        "contingency": contingency,
        "contingency_row_norm": contingency_row_norm,
        "contingency_col_norm": contingency_col_norm,
        "enrichment": enrich_df,
    }

In [ ]:
gene_embeddings = pd.DataFrame(
    z_bar[gene_ids],
    index=all_genes,
)

results = align_and_summarize(
    clusters_df=perts_aligned,          # gene↦cluster from covariance HC
    pathways_df=pathway_df,             # gene↦pathway ground truth
    gene_col="gene",
    cluster_col="cluster",
    pathway_col="pathway",
    min_pathway_size=1,                 # optional: filter tiny pathways
    gene_embeddings=gene_embeddings,
)

print("Best cluster→pathway mapping:")
for cl, pw in results["mapping"].items():
    print(f"  {cl} → {pw}")

print(f"\nGlobal alignment accuracy: {results['accuracy']:.3f}")
print(f"Precision: {results['precision']:.3f} | Recall: {results['recall']:.3f} | F1: {results['f1']:.3f}")
print(f"ARI: {results['ari']:.3f} | AMI: {results['ami']:.3f}")
print(f"Homogeneity: {results['homogeneity']:.3f} | Completeness: {results['completeness']:.3f} | V-measure: {results['v_measure']:.3f}")
print(f"Silhouette:  {results['silhouette']:.3f}")

print("\nPer-cluster purity:")
print(results["purity_df"])

In [ ]:
ev_df = slk.pl.make_ev_long_df(data)


merged = results["merged"].copy()

path_map = merged[["gene", "pathway", "mapped_pathway", "is_match"]].rename(
    columns={"gene": "pert"}
)

ev_annot = ev_df.merge(path_map, on="pert", how="inner")

ev_annot["pathway_used"] = ev_annot["mapped_pathway"].astype(str)

alpha = 0.05
ev_sig = ev_annot[ev_annot["qval"] < alpha].copy()
if ev_sig.empty:
    print(f"No significant (pert, gene) pairs at q < {alpha}.")
else:
    gp = ev_sig.groupby(["pathway_used", "gene"])
    pathway_targets = (
        gp.agg(
            n_perts=("pert", "nunique"),
            mean_ev=("ev", "mean"),
            mean_qval=("qval", "mean"),
        )
        .reset_index()
        .rename(columns={"pathway_used": "pathway", "gene": "target_gene"})
        .sort_values(["pathway", "mean_qval", "target_gene"])
    )
    print(
        pathway_targets
    )

### Fig. 1f — pathway-level MSigDB Hallmark enrichment of downstream targets
*(source cell 40)*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import gseapy as gp


def run_group_enrichment_hallmark(
    pathway_targets: pd.DataFrame,
    min_genes_per_group: int = 5,
    max_genes_per_group: int = 200,
    hallmark_lib: str = "MSigDB_Oncogenic_Signatures",  # C6 here
    organism: str = "Human",
    top_terms_per_group: int = 5,
    adj_alpha: float = 0.05,   # <-- significance cutoff on Adjusted P-value
):
    """
    For each internal group/pathway, run enrichment on its target genes
    and return a tidy table of *significant* enriched terms.

    Expected columns in pathway_targets:
      - 'pathway'     : internal label (EXOSOME, SPLICEOSOME, ...)
      - 'target_gene' : gene symbol
      - (optional) 'mean_ev' : used to sort genes before truncating
    """
    df = pathway_targets.copy()

    # Drop 'None' groups if present
    df = df[df["pathway"] != "None"].copy()

    # Sort within each pathway by mean_ev if present (strongest targets first)
    if "mean_ev" in df.columns:
        df = df.sort_values(["pathway", "mean_ev"], ascending=[True, False])

    # Collect gene lists per pathway
    path_to_genes = (
        df.groupby("pathway")["target_gene"]
          .apply(lambda s: list(dict.fromkeys(s.astype(str))))  # drop dups, keep order
          .to_dict()
    )

    rows = []

    for pw, genes in path_to_genes.items():
        # Cap number of genes (to avoid huge lists)
        genes = genes[:max_genes_per_group]

        if len(genes) < min_genes_per_group:
            continue

        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=[hallmark_lib],
            organism=organism,
            outdir=None,
            cutoff=1.0,  # keep all; we'll filter ourselves
        )

        res = enr.results.copy()
        if res.empty:
            continue

        res["pathway"] = pw

        # Parse overlap "k/n" -> k
        def parse_k(s):
            try:
                k, n = s.split("/")
                return int(k)
            except Exception:
                return np.nan

        res["overlap_k"] = res["Overlap"].astype(str).apply(parse_k)
        res["minus_log10_adjP"] = -np.log10(
            res["Adjusted P-value"].clip(lower=1e-300)
        )

        # === keep only significant terms ===
        res = res[res["Adjusted P-value"] <= adj_alpha]
        if res.empty:
            continue

        # Take top N significant terms by adjusted p-value
        res = res.sort_values("Adjusted P-value").head(top_terms_per_group)

        rows.append(
            res[
                ["pathway", "Term", "Adjusted P-value",
                 "Overlap", "overlap_k", "minus_log10_adjP"]
            ]
        )

    if not rows:
        raise ValueError("No significant enrichment results at adj_alpha "
                         f"= {adj_alpha}. Try relaxing the cutoff or min_genes_per_group.")

    enrich_df = pd.concat(rows, ignore_index=True)
    return enrich_df


def plot_hallmark_dotplot(enrich_df: pd.DataFrame, title: str = ""):
    """
    Dot plot:
      - y-axis: internal groups (pathway)
      - x-axis: enriched terms
      - size: number of overlapping genes (overlap_k)
      - color: -log10(adjusted p-value)
    """
    df = enrich_df.copy()

    # Categorical encodings for axes
    path_cats = pd.Categorical(df["pathway"])
    term_cats = pd.Categorical(df["Term"])

    df["y"] = path_cats.codes
    df["x"] = term_cats.codes

    # Sizes based on overlap_k
    k = df["overlap_k"].fillna(0).astype(float)
    if k.max() > 0:
        k_norm = (k - k.min()) / (k.max() - k.min() + 1e-8)
    else:
        k_norm = k
    size_min, size_max = 50, 600  # points^2
    df["dot_size"] = size_min + k_norm * (size_max - size_min)

    # Colors based on -log10(adjP)
    cvals = df["minus_log10_adjP"]
    norm = mpl.colors.Normalize(vmin=cvals.min(), vmax=cvals.max())
    cmap = plt.cm.viridis

    fig, ax = plt.subplots(
        figsize=(0.5 * len(term_cats.categories) + 4,
                 0.4 * len(path_cats.categories) + 3)
    )

    sc = ax.scatter(
        df["x"],
        df["y"],
        s=df["dot_size"],
        c=cvals,
        cmap=cmap,
        norm=norm,
        edgecolor="k",
        linewidth=0.3,
        alpha=0.9,
    )

    ax.set_xticks(np.arange(len(term_cats.categories)))
    ax.set_xticklabels(term_cats.categories, rotation=90, ha="right")

    ax.set_yticks(np.arange(len(path_cats.categories)))
    ax.set_yticklabels(path_cats.categories)

    ax.set_xlabel("Enriched signature")
    ax.set_ylabel("Internal group / pathway")

    ax.set_title(title)

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("-log10(Adjusted P-value)")

    plt.tight_layout()
    plt.show()



enrich = run_group_enrichment_hallmark(
    pathway_targets=pathway_targets,
    min_genes_per_group=5,
    max_genes_per_group=200,
    hallmark_lib="MSigDB_Hallmark_2020",
    organism="Human",
    top_terms_per_group=20,
    adj_alpha=0.05,   
)

plot_hallmark_dotplot(enrich, title="Pathway-level MSigDB_Hallmark_2020 enrichment of downstream targets")
